Sascha Spors,
Professorship Signal Theory and Digital Signal Processing,
Institute of Communications Engineering (INT),
Faculty of Computer Science and Electrical Engineering (IEF),
University of Rostock,
Germany

# Data Driven Audio Signal Processing - A Tutorial with Computational Examples

Winter Semester 2025/26 (Master Course #24512)

- lecture: https://github.com/spatialaudio/data-driven-audio-signal-processing-lecture
- tutorial: https://github.com/spatialaudio/data-driven-audio-signal-processing-exercise

Feel free to contact lecturer frank.schultz@uni-rostock.de

# TensorFlow Model for Binary Logistic Regression with One Sigmoid Layer
- TensorFlow is used to train the model and to make predictions
- data synthesis and data split is done with `binary_log_reg_toy_data()` which uses scikit-learn 
- statistical measures are calculated with scikit-learn
- the implementation uses 64-Bit double precision
- manual initialisation of model weights
- no shuffling of batch data -> vanilla batch gradient descent
- static learning rate
- hence, the training is **fully deterministic** and thus all results are precisely identical with those from other exemplary implementations
    - [binary_logistic_regression_manual.ipynb](binary_logistic_regression_manual.ipynb)
    - [binary_logistic_regression_torch.ipynb](binary_logistic_regression_torch.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sklearn

from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn.metrics import balanced_accuracy_score, accuracy_score

import tensorflow as tf
from tensorflow import keras

from util_binary_logistic_regression import toy_data, init_weights
from util_binary_logistic_regression import my_sigmoid, predict_class

tf.keras.backend.set_floatx("float64")

# last manual check with ('2.20.0', '3.11.3', '1.7.2')
tf.__version__, keras.__version__, sklearn.__version__


## Data

In [ ]:
X_train, Y_train, X_test, Y_test = toy_data()
N_Features = X_train.shape[1]

## Learning Parameters

In [ ]:
batch_size = X_train.shape[0] // 400
num_epochs = 10
learning_rate = 0.1

## Prepare Data for TensorFlow

In [ ]:
Y_train = Y_train[:, None]
Y_test = Y_test[:, None]

## Define TensorFlow Model

In [ ]:
initializer = keras.initializers.RandomUniform(
    minval=0.0,
    maxval=1.0)
input = keras.Input(shape=(N_Features, ))
output = keras.layers.Dense(
    1, kernel_initializer=initializer,
    activation='sigmoid')(input)
model = keras.Model(
    inputs=input,
    outputs=output)

## Define Loss Function and Optimizer Strategy

In [ ]:
loss = keras.losses.BinaryCrossentropy(
    from_logits=False,
    label_smoothing=0)
metrics = [keras.metrics.BinaryCrossentropy()]
optimizer = keras.optimizers.SGD(
    learning_rate=learning_rate)

## Init Model Parameters
to obtain reproducible results with the other implementations

In [ ]:
# init the model parameters (2 weights and 1 bias)
w1, w2, b = init_weights()
model.set_weights([np.array([[w1], [w2]]), np.array([b])])
print(model.get_weights())

## Compile the Model

In [ ]:
model.compile(
    optimizer=optimizer,
    loss=loss,
    metrics=metrics)
print(model.summary())

## Train the Model

In [ ]:
model_log = model.fit(
    X_train, Y_train,
    batch_size=batch_size,
    shuffle=False,
    epochs=num_epochs,
    validation_data=(X_test, Y_test),
    verbose=3)
for i in range(len(model_log.history['loss'])):
    print('epoch:', i+1)
    print('empirical risk train:',
          '%0.15e' % model_log.history['loss'][i],
          ', empirical risk test:',
          '%0.15e' % model_log.history['val_loss'][i])

## Check Model Parameters

In [ ]:
tmp = model.get_weights()
print('%+0.15e' % tmp[0][0, 0], '\n%+0.15e' % tmp[0][1, 0])
print('%+0.15e' % tmp[1][0])

## Model Test

### Empirical Risk

In [ ]:
loss_train = model.evaluate(
    X_train, Y_train, verbose=0)
print('empirical risk train:', '%0.15e' % loss_train[0])

loss_test = model.evaluate(
    X_test, Y_test, verbose=0)
print('empirical risk test: ', '%0.15e' % loss_test[0])

### Prep for Class Prediction Metrics

In [ ]:
# from here, we work with numpy rank 1 arrays, i.e. (8000,) and (2000,)
y_true_train = Y_train[:, 0]
y_pred_train = predict_class(model.predict(X_train, verbose=0))[:, 0]
y_true_test = Y_test[:, 0]
y_pred_test = predict_class(model.predict(X_test, verbose=0))[:, 0]

### Confusion Matrix

In [ ]:
print('confusion matrix train absolute')
print(confusion_matrix(
    y_true_train,
    y_pred_train,
    normalize=None))
print('confusion matrix train in %')
print(confusion_matrix(
    y_true_train,
    y_pred_train,
    normalize='all')*100)
print('\nconfusion matrix test absolute')
print(confusion_matrix(
    y_true_test,
    y_pred_test,
    normalize=None))
print('confusion matrix test in %')
print(confusion_matrix(
    y_true_test,
    y_pred_test,
    normalize='all')*100)

### Precision, Recall, F1Score, Support

In [ ]:
p, r, f, s = precision_recall_fscore_support(
    y_true_train, y_pred_train)
print(p, r, f, s)
p, r, f, s = precision_recall_fscore_support(
    y_true_test, y_pred_test)
print(p, r, f, s)

### Accuracy, Balanced Accuracy

We have a very balanced data set, hence both values are very similar

In [ ]:
a = accuracy_score(
    y_true_train, y_pred_train)
ba = balanced_accuracy_score(
    y_true_train, y_pred_train)
print(a, ba)

a = accuracy_score(
    y_true_test, y_pred_test)
ba = balanced_accuracy_score(
    y_true_test, y_pred_test)
print(a, ba)

## Plot Data Points and Decision Plane

In [ ]:
# get model parameters
weights = model.get_weights()
w, b = weights[0].T, weights[1]

# get probabilities in the prediction plane
levels = [0.0, 0.05, 0.1, 0.37, 0.5, 0.63, 0.9, 0.95, 1]
f1, f2 = np.arange(-5, 5, 0.05), np.arange(-5, 5, 0.05)
xv, yv = np.meshgrid(f1, f2)
# the model prediction as manual one-liner, this yields a probability
prob_plane = my_sigmoid(w[0, 0] * xv + w[0, 1] * yv + b)
# hard decision boundary for classes 0,1:
# prob_plane = (prob_plane>=0.5)*1

plt.figure(figsize=(10, 10))
plt.subplot(2, 2, 1)
plt.plot(X_train[Y_train[:, 0] == 1, 0],
         X_train[Y_train[:, 0] == 1, 1],
         "o", color='orangered', ms=1)
plt.contourf(f1, f2, prob_plane, levels=levels, cmap="RdBu_r")
plt.axis("equal")
plt.colorbar()
plt.title("training class '1' " + str(X_train.shape))
plt.xlabel("feature 1")
plt.ylabel("feature 2")

plt.subplot(2, 2, 2)
plt.plot(X_train[Y_train[:, 0] == 0, 0],
         X_train[Y_train[:, 0] == 0, 1],
         "o", color='dodgerblue', ms=1)
plt.contourf(f1, f2, prob_plane, levels=levels, cmap="RdBu_r")
plt.axis("equal")
plt.colorbar()
plt.title("training class '0' " + str(X_train.shape))
plt.xlabel("feature 1")
plt.ylabel("feature 2")

plt.subplot(2, 2, 3)
plt.plot(X_test[Y_test[:, 0] == 1, 0],
         X_test[Y_test[:, 0] == 1, 1],
         "o", color='orangered', ms=1)
plt.contourf(f1, f2, prob_plane, levels=levels, cmap="RdBu_r")
plt.axis("equal")
plt.colorbar()
plt.title("test class '1' " + str(X_test.shape))
plt.xlabel("feature 1")
plt.ylabel("feature 2")

plt.subplot(2, 2, 4)
plt.plot(X_test[Y_test[:, 0] == 0, 0],
         X_test[Y_test[:, 0] == 0, 1],
         "o", color='dodgerblue', ms=1)
plt.contourf(f1, f2, prob_plane, levels=levels, cmap="RdBu_r")
plt.axis("equal")
plt.colorbar()
plt.title("test class '0' " + str(X_test.shape))
plt.xlabel("feature 1")
plt.ylabel("feature 2")

## Copyright

- the notebooks are provided as [Open Educational Resources](https://en.wikipedia.org/wiki/Open_educational_resources)
- feel free to use the notebooks for your own purposes
- the text is licensed under [Creative Commons Attribution 4.0](https://creativecommons.org/licenses/by/4.0/)
- the code of the IPython examples is licensed under the [MIT license](https://opensource.org/licenses/MIT)
- please attribute the work as follows: *Frank Schultz, Data Driven Audio Signal Processing - A Tutorial Featuring Computational Examples, University of Rostock* ideally with relevant file(s), github URL https://github.com/spatialaudio/data-driven-audio-signal-processing-exercise, commit number and/or version tag, year.